In [ ]:
import sys
sys.path.append('..')
import os
import sklearn.metrics.cluster
import sklearn.cluster
import seaborn as sns
import scanpy as sc
import sklearn as sk
from sklearn.neighbors import NearestNeighbors
import matplotlib as mpl
from matplotlib import pyplot as plt
import skdim
import seaborn as sns
import time
import numpy as np
import pandas as pd
import pickle
import rpy2.robjects as ro
import rpy2.robjects.numpy2ri
import rpy2.robjects.packages as rpackages
import utils
import anndata2ri
import gc
import benchmark_common_functions
from itertools import product

anndata2ri.activate()
ro.numpy2ri.activate()
readRDS = ro.r['readRDS']
%load_ext rpy2.ipython

In [ ]:
rootdir = '../data/DuoClustering2018/sce_full/'
resdir = '../results/DuoClustering2018/sce_full/'
os.makedirs(resdir, exist_ok=True)

csv_candidates = [
    f for f in os.listdir(rootdir)
    if f.startswith('sce_full_') and f.endswith('.csv')
    and not f.endswith('_labels.csv') and not f.endswith('_log.csv')
]
fnames = sorted(
    f for f in csv_candidates
    if os.path.exists(rootdir+f[:-4]+'_labels.csv')
)

#get_res_path = lambda fname: f'{resdir}{fname}_norm{do_norm}_scale{norm_scale}_log{do_log}_pca{do_pca}_ncomps{n_comps}_{metric}_weighted{weighted}_knn{n_neighbors}_seed{seed}.pkl'
#get_res_path_orig = lambda fname: f'{resdir}{fname}_orig_pca{do_pca}_ncomps{n_comps}_{metric}_weighted{weighted}_knn{n_neighbors}_seed{seed}.pkl'

get_res_path = lambda fname: f'{resdir}{fname}_norm{do_norm}_scale{norm_scale}_log{do_log}_pca{do_pca}_ncomps{n_comps}_{metric}_weighted{weighted}_knn_sqrt_seed{seed}.pkl'
get_res_path_orig = lambda fname: f'{resdir}{fname}_orig_pca{do_pca}_ncomps{n_comps}_{metric}_weighted{weighted}_knn_sqrt_seed{seed}.pkl'
get_res_path_sc = lambda fname: f'{resdir}{fname}_norm{do_norm}_scale{norm_scale}_log{do_log}_pca{do_pca}_ncomps{n_comps}_{metric}_weighted{weighted}_knn_sqrt_seed{seed}_scanpyhubness.pkl'


# Run all datasets original

In [ ]:
#fixed params
do_norm = False
norm_scale = False
do_log = False
do_pca = True
n_neighbors = 10
seed = 0

#vary params
metric=('cosine','euclidean')
n_comps=(25,50,100,500)
weighted=(True,)

params_list = list(product(metric,n_comps,weighted))

In [ ]:
for metric,n_comps,weighted in params_list:
    print(metric,n_comps,weighted)
    for fname in fnames:
        if get_res_path_orig(fname[:-4]).split('/')[-1] in os.listdir(resdir):
            continue
        print('\t\t'+fname)
        dataset = fname[:-4]
        adata = sc.read_csv(rootdir+dataset+'_log.csv').transpose()
        adata.obs['phenoid'] = pd.read_csv(
            rootdir+dataset+'_labels.csv'
        ).iloc[:,1].values

        results_dict = benchmark_common_functions.clustering_analysis(
            adata,
            true_labels = adata.obs['phenoid'],
            do_norm = do_norm,
            norm_scale = norm_scale,
            do_log = do_log,
            do_pca = do_pca,
            n_neighbors=int(round(np.sqrt(len(adata)))),
            n_clusters= adata.obs['phenoid'].nunique(),
            metric=metric,
            weighted=weighted,  #weighted adjmat for louvain/leiden clustering ?
            seed=seed,
            n_comps=n_comps
        )

        with open(get_res_path_orig(fname[:-4]),'wb') as f:
            pickle.dump(results_dict,f)
    
        %xdel adata
        gc.collect()

# Run all datasets

In [ ]:
#fixed params
do_log = True
do_pca = True
n_neighbors = 10
seed = 0

#vary params
metric=('cosine','euclidean')
n_comps=(25,50,100,500)
weighted=(True,)
do_norm = ('duo','seurat')#,'zheng17')
norm_scale = (True,False)
params_list = list(product(metric,n_comps,weighted,do_norm,norm_scale))

In [ ]:
for metric,n_comps,weighted,do_norm,norm_scale in params_list:
    print(metric,n_comps,weighted,do_norm,norm_scale)
    for fname in fnames:
        if get_res_path(fname[:-4]).split('/')[-1] in os.listdir(resdir):
            continue
        print('\t\t'+fname)
        dataset = fname[:-4]
        adata = sc.read_csv(rootdir+fname).transpose()
        adata.obs['phenoid'] = pd.read_csv(
            rootdir+dataset+'_labels.csv'
        ).iloc[:,1].values
        results_dict = benchmark_common_functions.scanpy_hubness_analysis(
            adata,
            true_labels = adata.obs['phenoid'],
            do_norm = do_norm,
            norm_scale = norm_scale,
            do_log = do_log,
            do_pca = do_pca,
            n_neighbors=int(round(np.sqrt(len(adata)))),
            n_clusters= adata.obs['phenoid'].nunique(),
            metric=metric,
            weighted=weighted,  #weighted adjmat for louvain/leiden clustering ?
            seed=seed,
            n_comps=n_comps
        )

        with open(get_res_path(fname[:-4]),'wb') as f:
            pickle.dump(results_dict,f)
    
        %xdel adata
        %R rm(list = ls())
        %R gc()
        time.sleep(2)
        gc.collect()

In [ ]:
for metric,n_comps,weighted,do_norm,norm_scale in params_list:
    print(metric,n_comps,weighted,do_norm,norm_scale)
    for fname in fnames:
        if get_res_path_sc(fname[:-4]).split('/')[-1] in os.listdir(resdir):
            continue
        print('\t\t'+fname)
        dataset = fname[:-4]
        adata = sc.read_csv(rootdir+fname).transpose()
        adata.obs['phenoid'] = pd.read_csv(
            rootdir+dataset+'_labels.csv'
        ).iloc[:,1].values
        results_dict = benchmark_common_functions.scanpy_hubness_analysis(
            adata,
            do_norm = do_norm,
            norm_scale = norm_scale,
            do_log = do_log,
            do_pca = do_pca,
            n_neighbors=int(round(np.sqrt(len(adata)))),
            metric=metric,
            weighted=weighted,  #weighted adjmat for louvain/leiden clustering ?
            seed=seed,
            n_comps=n_comps
        )

        with open(get_res_path_sc(fname[:-4]),'wb') as f:
            pickle.dump(results_dict,f)
    
        %xdel adata
        %R rm(list = ls())
        %R gc()
        time.sleep(2)
        gc.collect()